# 04 - Chargement en base (staging + core)

Ce notebook illustre l'exécution manuelle de tout le pipeline (equivalent de ce que fait le DAG Airflow), utile pour tester avant de tout automatiser.

In [2]:
import sys, os
sys.path.append(os.path.abspath('../src'))

from db import get_engine
from extract import extract_and_load
from clean import clean_pipeline
from load import load_pipeline
from quality_check import run_quality_checks

engine = get_engine()

## 1. Extraction -> staging

In [3]:
df_raw = extract_and_load('../data/raw/Sample - Superstore.csv', engine)

[extract] 10064 lignes chargées dans staging.superstore_raw


## 2. Nettoyage

In [4]:
df_clean = clean_pipeline(df_raw)

[clean] 70 doublon(s) supprimé(s) (sur row_id)
[clean] 298 ligne(s) supprimée(s) pour clés/dates manquantes obligatoires
[clean] 25 ligne(s) invalide(s) supprimée(s) (discount>100% / quantity<0 / ship_date<order_date)
[clean] total lignes supprimées à cette étape : 216


## 3. Chargement -> core (idempotent, via UPSERT)

In [5]:
stats = load_pipeline(df_clean, engine)
stats

[load] 793 ligne(s) upsertées dans core.customers
[load] 1856 ligne(s) upsertées dans core.products
[load] 9480 ligne(s) upsertées dans core.orders


{'nb_customers': 793, 'nb_products': 1856, 'nb_orders': 9480}

## 4. Contrôle qualité final

In [6]:
checks = run_quality_checks(engine)
checks

[quality_check] Tous les contrôles sont OK : {'nb_customers': 793, 'nb_products': 1856, 'nb_orders': 9480, 'nb_orders_sans_fk': 0, 'nb_noms_non_hashes': 0, 'nb_orders_invalides': 0}


{'nb_customers': 793,
 'nb_products': 1856,
 'nb_orders': 9480,
 'nb_orders_sans_fk': 0,
 'nb_noms_non_hashes': 0,
 'nb_orders_invalides': 0}

## 5. Statistiques de restitution demandées par le brief

In [7]:
import pandas as pd

print('Nombre total de clients  :', stats['nb_customers'])
print('Nombre total de produits :', stats['nb_products'])
print('Nombre total de commandes:', stats['nb_orders'])

repartition = pd.read_sql(
    '''
    SELECT p.category, c.region, c.segment, SUM(o.sales) AS total_sales
    FROM core.orders o
    JOIN core.products p ON o.productid = p.productid
    JOIN core.customers c ON o.customerid = c.customerid
    GROUP BY p.category, c.region, c.segment
    ORDER BY total_sales DESC;
    ''',
    con=engine,
)
repartition.head(10)

Nombre total de clients  : 793
Nombre total de produits : 1856
Nombre total de commandes: 9480


,category,region,segment,total_sales
0,Office Supplies,West,Consumer,3521133.20
1,Furniture,East,Consumer,2331642.58
2,Furniture,West,Consumer,1279598.89
3,Office Supplies,Central,Consumer,1206612.14
4,Technology,East,Corporate,1200864.17
5,Furniture,Central,Corporate,1188378.98
6,Technology,East,Home Office,1187346.70
7,Office Supplies,Central,Corporate,1181987.06
8,Furniture,West,Home Office,1169833.37
9,Office Supplies,South,Corporate,1165492.81


## Test d'idempotence

Relance simplement les 3 cellules d'extraction/nettoyage/chargement ci-dessus une deuxième fois : les compteurs (`nb_customers`, `nb_products`, `nb_orders`) doivent rester IDENTIQUES, preuve qu'aucun doublon n'a été ajouté grâce à la logique UPSERT (`ON CONFLICT DO UPDATE`).